In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import io
from transformers import AutoTokenizer, AutoModel
from collatex import Collation, collate
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
lemmatizer = WordNetLemmatizer()

def get_distilbert_embeddings(texts):
    """Extracts [CLS] token embeddings using DistilBERT"""
    print("    Loading DistilBERT (distilbert-base-uncased)...")
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    embeddings = []
    batch_size = 32
    
    texts = list(texts)
    
    print(f"    Computing embeddings on {device}...")
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt", max_length=64)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(batch_embeddings)
            
    return np.vstack(embeddings).astype(np.float32)


def get_collatex_story_features(df):
    story_feats = []
    for story_id, group in df.groupby('STORY'):
        if len(group) < 2: continue
        collation = Collation()
        for i, (_, row) in enumerate(group.iterrows()):
            t = str(row['TITLE']).strip()
            if len(t) < 3: continue
            collation.add_plain_witness(f"w{i}", t)
        try:
            csv_str = collate(collation, output='csv')
            align = pd.read_csv(io.StringIO(csv_str), index_col=0)
            counts = {'same':0, 'lexical':0, 'morphological':0, 'plus/minus':0}
            for col in align.columns:
                toks = [str(v).strip() for v in align[col] if pd.notna(v) and str(v).strip()!='']
                if len(toks) <= 1: counts['same'] += 1; continue
                unique = list(set(toks))
                lemmas = [lemmatizer.lemmatize(w.lower()) for w in unique]
                if len(set(lemmas)) == 1: counts['morphological'] += 1
                elif any(t.startswith(('+','-')) or t.replace('.','',1).isdigit() for t in toks): counts['plus/minus'] += 1
                else: counts['lexical'] += 1
            total = sum(counts.values())
            if total > 0:
                story_feats.append({'STORY': story_id, **{k: v/total for k,v in counts.items()}})
        except: pass
    return pd.DataFrame(story_feats) if story_feats else pd.DataFrame(columns=['STORY','same','lexical','morphological','plus/minus'])

def get_tfidf(texts, analyzer='word', ngram_range=(1,2), max_feat=2500):
    vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, max_features=max_feat, sublinear_tf=True)
    return vec.fit_transform(texts).toarray()


def run_distilbert_experiment(filepath, dataset_name, feb_baseline_f1):
    print(f"\n{'='*70}")
    print(f" DISTILBERT HYBRID PIPELINE - {dataset_name.upper()}")
    print(f"{'='*70}")
    
    df = pd.read_csv(filepath)
    
    print("🧹 Cleaning data...")
    df = df.dropna(subset=['TITLE'])
    df['TITLE'] = df['TITLE'].astype(str)
    df = df[df['TITLE'].str.strip() != ''].copy()
    
    texts = df['TITLE'].values
    labels = df['PUBLISHER'].values
    print(f" Loaded {len(texts)} valid articles.")
    
    X_distil = get_distilbert_embeddings(texts)
    
    print(" Extracting CollateX variant features...")
    collatex_df = get_collatex_story_features(df)
    df = df.merge(collatex_df, on='STORY', how='left').fillna(0)
    X_collatex = df[['same', 'lexical', 'morphological', 'plus/minus']].values
    
    print(" Extracting TF-IDF Char n-grams...")
    X_tfidf_c = get_tfidf(texts, analyzer='char', ngram_range=(2,4))
    
    print(" Combining features: [DistilBERT] + [CollateX] + [TF-IDF Char]")
    X_hybrid = np.hstack([X_distil, X_collatex, X_tfidf_c])
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        'Naive Bayes': GaussianNB(),
        'SVM': LinearSVC(random_state=42, max_iter=2000, class_weight='balanced'),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'SGD': SGDClassifier(loss='log_loss', random_state=42, max_iter=1000, class_weight='balanced')
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    
    print(" Training with 5-Fold CV...\n")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_hybrid)
    
    for model_name, model in models.items():
        try:
            acc = cross_val_score(model, X_scaled, labels, cv=cv, scoring='accuracy').mean()
            f1 = cross_val_score(model, X_scaled, labels, cv=cv, scoring='f1_macro').mean()
            results.append({'Model': model_name, 'Accuracy': acc, 'F1_Macro': f1})
            print(f"   {model_name:<20} | Acc: {acc:.4f} | F1: {f1:.4f}")
        except Exception as e:
            print(f"   {model_name} failed: {str(e)[:50]}...")
    
    res_df = pd.DataFrame(results)
    best_f1 = res_df['F1_Macro'].max()
    
    print(f"\n BEST F1 for {dataset_name.upper()}: {best_f1:.4f}")
    print(f" Improvement over Feb Baseline ({feb_baseline_f1:.4f}): {best_f1 - feb_baseline_f1:+.4f}")
    
    del X_hybrid, X_distil, X_tfidf_c
    gc.collect()
    return res_df


duo_res = run_distilbert_experiment('clean_duo_data.csv', 'DUO', feb_baseline_f1=0.882)
trio_res = run_distilbert_experiment('clean_trio_data.csv', 'TRIO', feb_baseline_f1=0.788)

print("\n" + "="*50)
print(" FINAL DISTILBERT RESULTS SUMMARY")
print("="*50)
print(f"DUO  | Best Model: {duo_res.loc[duo_res['F1_Macro'].idxmax(), 'Model']} | F1: {duo_res['F1_Macro'].max():.4f}")
print(f"TRIO | Best Model: {trio_res.loc[trio_res['F1_Macro'].idxmax(), 'Model']} | F1: {trio_res['F1_Macro'].max():.4f}")


🔬 DISTILBERT HYBRID PIPELINE - DUO
🧹 Cleaning data...
✅ Loaded 3073 valid articles.
   🤖 Loading DistilBERT (distilbert-base-uncased)...
   🚀 Computing embeddings on cpu...
📊 Extracting CollateX variant features...
📝 Extracting TF-IDF Char n-grams...
🔗 Combining features: [DistilBERT] + [CollateX] + [TF-IDF Char]
🚀 Training with 5-Fold CV...

  ✅ Logistic Regression  | Acc: 0.8737 | F1: 0.8647
  ✅ Naive Bayes          | Acc: 0.7872 | F1: 0.7854
  ✅ SVM                  | Acc: 0.8575 | F1: 0.8491
  ✅ KNN                  | Acc: 0.8083 | F1: 0.7877
  ✅ SGD                  | Acc: 0.8594 | F1: 0.8435

🏆 BEST F1 for DUO: 0.8647
📈 Improvement over Feb Baseline (0.8820): -0.0173

🔬 DISTILBERT HYBRID PIPELINE - TRIO
🧹 Cleaning data...
✅ Loaded 2946 valid articles.
   🤖 Loading DistilBERT (distilbert-base-uncased)...
   🚀 Computing embeddings on cpu...
📊 Extracting CollateX variant features...
📝 Extracting TF-IDF Char n-grams...
🔗 Combining features: [DistilBERT] + [CollateX] + [TF-IDF Char]
